# Gradient Boosting und Modellvergleich

Boosting baut kleine Bäume nacheinander. Jeder neue Baum korrigiert den aktuellen Loss; der Random Forest trainiert seine Bäume dagegen unabhängig.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.ensemble import GradientBoostingClassifier,RandomForestClassifier
from sklearn.metrics import accuracy_score,f1_score,log_loss,precision_score,recall_score,roc_auc_score
from sklearn.model_selection import GridSearchCV,train_test_split
from sklearn.tree import DecisionTreeClassifier
X,y=make_classification(n_samples=1100,n_features=10,n_informative=5,n_redundant=2,weights=[.65,.35],class_sep=.9,flip_y=.04,random_state=42)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,random_state=42,stratify=y)

def metriken(name, modell, X_test, y_test):
    pred = modell.predict(X_test)
    prob = modell.predict_proba(X_test)[:, 1]
    return {"Modell": name, "Accuracy": accuracy_score(y_test,pred),
            "Precision": precision_score(y_test,pred), "Recall": recall_score(y_test,pred),
            "F1": f1_score(y_test,pred), "ROC-AUC": roc_auc_score(y_test,prob)}

## Lernverlauf und Hyperparameter

`learning_rate` gewichtet jede Korrektur und muss mit `n_estimators` abgestimmt werden. `max_depth` ist häufig nur 1 bis 3. `min_samples_leaf` glättet, `subsample < 1` erzeugt stochastisches Boosting, `max_features` begrenzt Merkmale. `n_iter_no_change` ermöglicht Early Stopping.

In [ ]:
lang=GradientBoostingClassifier(n_estimators=300,learning_rate=.08,max_depth=2,random_state=42).fit(X_train,y_train)
kurve=[]
for i,(a,b) in enumerate(zip(lang.staged_predict_proba(X_train),lang.staged_predict_proba(X_test)),1): kurve.append({"Bäume":i,"Training":log_loss(y_train,a),"Test":log_loss(y_test,b)})
pd.DataFrame(kurve).plot(x="Bäume",figsize=(9,4)); plt.ylabel("Log-Loss"); plt.grid(alpha=.3); plt.show()

## Kleine Suche für alle drei Verfahren

Gleicher Split, gleiche Folds und F1-Metrik machen den Vergleich fair. Die Parameterbereiche bleiben bewusst klein und modellspezifisch.

In [ ]:
suchen={
"Entscheidungsbaum":GridSearchCV(DecisionTreeClassifier(random_state=42),{"max_depth":[3,6,None],"min_samples_leaf":[1,5,15]},scoring="f1",cv=4,n_jobs=-1),
"Random Forest":GridSearchCV(RandomForestClassifier(random_state=42,n_jobs=-1),{"n_estimators":[100,250],"max_depth":[6,None],"min_samples_leaf":[1,4]},scoring="f1",cv=4,n_jobs=-1),
"Gradient Boosting":GridSearchCV(GradientBoostingClassifier(random_state=42),{"n_estimators":[100,200],"learning_rate":[.03,.1],"max_depth":[1,2,3],"subsample":[.8,1.]},scoring="f1",cv=4,n_jobs=-1)}
result=[]
for name,s in suchen.items():
    start=time.perf_counter(); s.fit(X_train,y_train); row=metriken(name,s.best_estimator_,X_test,y_test); row.update({"CV-F1":s.best_score_,"Suchzeit_s":time.perf_counter()-start,"Beste Parameter":s.best_params_}); result.append(row)
vergleich=pd.DataFrame(result).set_index("Modell"); display(vergleich.round(3)); vergleich[["F1","ROC-AUC","CV-F1"]].plot.bar(figsize=(10,5),ylim=(.7,1)); plt.xticks(rotation=0); plt.show()

In [ ]:
pd.DataFrame({n:s.best_estimator_.feature_importances_ for n,s in suchen.items()},index=[f"Merkmal {i}" for i in range(10)]).plot.bar(figsize=(12,5)); plt.title("Feature Importances im Vergleich"); plt.show()

**Einordnung:** Der Einzelbaum ist am vollständigsten erklärbar. Random Forest ist robust und parallelisierbar. Gradient Boosting kann sehr leistungsfähig sein, reagiert aber empfindlicher auf Parameter. Ein kleiner Score-Unterschied allein ersetzt keine Betrachtung von Laufzeit, Fehlkosten und Stabilität.